# How to Align a Piano Roll (SUPRA)

This tutorial applies everything we've learned to a real-world case study:
aligning a **scanned piano roll image** with **MIDI files**, **audio**, and
**score annotations** from the Stanford University Piano Roll Archive (SUPRA).

**Learning Objectives:**
- Create a graphical timeline from ATON analysis data
- Add C-Maps for physical unit conversion (pixels to inches/cm)
- Use the **child timeline** API for hierarchical relationships
- Load MIDI, audio, and score data with specialized loaders
- Build a **TimelineGroup** connecting all representations
- Transfer coordinates across the entire alignment chain using **Timestamps**

**Prerequisites:**
- Notebook 04 (Timelines, hierarchies)
- Notebook 05 (Timestamps)
- Notebook 07 (Alignment Basics, TimelineGroup)

**Data Source:**
- **Roll**: WM 990 (Welte-Mignon red roll, T-100)
- **Piece**: Richard Wagner - Meistersinger von Nurnberg: Vorspiel (Prelude)
- **Performer**: Myrtle Elvyn, piano (December 6, 1905)
- **SUPRA URL**: https://supra.stanford.edu/
- **DRUID**: fd660zf8362

## Gold Standard Reference Values

Per the ZERO TOLERANCE policy, all values use exact counts from the SUPRA
analysis:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `IMAGE_WIDTH` | 4,096 | Image width in pixels |
| `IMAGE_HEIGHT` | 299,400 | Image height in pixels |
| `LENGTH_DPI` | 300.25 | Scan resolution (pixels per inch) |
| `PHYSICAL_LENGTH` | 997.17 | Roll length in inches (25.33 m) |
| `FIRST_HOLE` | 15,343 | Pixel row of first musical hole |
| `LAST_HOLE` | 293,119 | Pixel row of last musical hole |
| `MUSICAL_LENGTH` | 277,776 | Pixels from first to last hole |
| `MUSICAL_HOLES` | 30,092 | Individual hole punches (raw MIDI events) |
| `RAW_MIDI_NOTES` | 30,092 | Raw MIDI note count (1 per hole) |
| `EXP_MIDI_NOTES` | 6,380 | Expressive MIDI note count (merged) |
| `SCORE_NOTES` | 5,577 | Notes in DCML score |
| `SCORE_MEASURES` | 222 | Measures in DCML score |
| `SCORE_LENGTH` | 888 | Total quarterbeats |

## Setup

In [1]:

from timetoalign import IdCoordinate, Ms3Loader, TimelineGroup, TimeUnit
from timetoalign.core import timestamp_table_to_dataframe
from timetoalign.loader.graphical.aton import ATONLoader
from timetoalign.loader.midi import PerformanceMidiLoader
from timetoalign.loader.physical import AudioLoader
from timetoalign.maps import ScalarMap
from timetoalign.testdata import ensure_data
from timetoalign.timelines import ContinuousPhysicalTimeline

DATA_DIR = ensure_data("supra")

ATON_FILE = DATA_DIR / "image" / "fd660zf8362_analysis.txt"
MIDI_RAW_PATH = DATA_DIR / "midi" / "fd660zf8362_raw.mid"
MIDI_EXP_PATH = DATA_DIR / "midi" / "fd660zf8362_exp.mid"
MP3_PATH = DATA_DIR / "midi" / "fd660zf8362.mp3"
DCML_DIR = DATA_DIR / "dcml"

{"Data directory": str(DATA_DIR), "ATON file": ATON_FILE.name}

{'Data directory': '/home/laser/git/tta/timetoalign/tests/data/supra',
 'ATON file': 'fd660zf8362_analysis.txt'}

***

## Part A: Create the Image Timeline (DGT1) from ATON Loader

The ATON (Artistic Text-based Object Notation) file contains hole punch data
from the Stanford SUPRA project's piano roll analysis. The loader creates a
timeline with **all hole events already populated** at their absolute pixel
coordinates.

**Note:** We use the `from_file()` constructor for one-line instantiation.

In [2]:
aton_loader = ATONLoader.from_file(ATON_FILE)

assert aton_loader.musical_holes == 30092
assert aton_loader.musical_notes == 8718
assert aton_loader.first_hole.value == 15343
assert aton_loader.last_hole.value == 293119
assert aton_loader.musical_length.value == 277776
assert aton_loader.image_dimensions["height"] == 299400

{
    "Image height": f"{aton_loader.image_dimensions['height']:,} pixels",
    "Musical holes": f"{aton_loader.musical_holes:,}",
    "First hole": aton_loader.first_hole,
    "Last hole": aton_loader.last_hole,
    "Musical length": aton_loader.musical_length,
    "Verification": "PASSED",
}

{'Image height': '299,400 pixels',
 'Musical holes': '30,092',
 'First hole': Coordinate(15343, pixels),
 'Last hole': Coordinate(293119, pixels),
 'Musical length': Coordinate(277776, pixels),
 'Verification': 'PASSED'}

The loader's `create_timeline()` method creates a timeline spanning the full
image with holes at absolute coordinates.

In [3]:
dgt1 = aton_loader.create_timeline(uid="dgt1", name="Piano Roll Image (WM 990)")
dgt1

DiscreteGraphicalTimeline(id='dgt1', length=299400, unit=pixels, events=30092, children=0)

***

## Part B: Add Physical Unit C-Maps

The piano roll was scanned at 300.25 DPI (dots per inch). We attach
ConversionMaps to convert pixel coordinates to physical units.

**Important:** We specify `name=` for human-readable column headers in
timestamp tables. Without it, columns would show as "map:ScalarMap_1".

**Calculation:** 299,400 pixels / 300.25 DPI = 997.17 inches = 25.33 meters

In [4]:
LENGTH_DPI = 300.25

# Note: name= provides readable column headers (defaults to "source_to_target")
dgt1.add_conversion_map(
    ScalarMap(
        scalar=1 / LENGTH_DPI,
        source_unit="pixels",
        target_unit="inches",
        name="pixels_to_inches",  # Human-readable name for timestamp columns
    )
)

dgt1.add_conversion_map(
    ScalarMap(
        scalar=2.54 / LENGTH_DPI,
        source_unit="pixels",
        target_unit="cm",
        name="pixels_to_cm",  # Human-readable name for timestamp columns
    )
)

The `convert_to` method returns proper Coordinate objects with units.

In [5]:
image_length_inches = dgt1.convert_to(dgt1.length, "inches")
image_length_cm = dgt1.convert_to(dgt1.length, "cm")

{
    "Image length": dgt1.length,
    "Image length (inches)": image_length_inches,
    "Image length (cm)": image_length_cm,
    "Image length (meters)": f"{image_length_cm.value / 100:.2f} meters",
}

{'Image length': Coordinate(299400, pixels),
 'Image length (inches)': Coordinate(997.1690258118235, inches),
 'Image length (cm)': Coordinate(2532.8093255620315, centimeters),
 'Image length (meters)': '25.33 meters'}

***

## Part C: Create Child Timeline for Relative Coordinates

The musical content doesn't span the entire image - holes start at pixel 15,343.
We create a **child timeline** to provide a relative coordinate view where the
first hole = 0 pixels.

**Key insight:** The child has no events of its own. It's purely a coordinate
transformation. When we get timestamps, we see event coordinates in BOTH
the parent (absolute) and child (relative) coordinate systems - for free!

In [6]:
dgt_holes = dgt1.create_child(
    length=aton_loader.musical_length,
    offset=aton_loader.first_hole,
    uid="dgt_holes",
    name="Musical Holes Region",
)
dgt1

DiscreteGraphicalTimeline(id='dgt1', length=299400, unit=pixels, events=30092, children=1, cmaps=2)

***

## Part D: Demonstrate Timestamps

With the parent-child hierarchy established, we can generate **timestamps**
that show coordinates in both the parent (full image) and child (holes region)
coordinate systems simultaneously.

**Note:** The `to_dataframe()` method provides column names with units appended
(e.g., "pixels_to_inches (inches)") and proper integer types. This is the
recommended way to get timestamp data for display.

In [7]:
timestamps_df = dgt1.to_dataframe()

{"Total timestamps": len(timestamps_df), "Columns": list(timestamps_df.columns)}

{'Total timestamps': 20680,
 'Columns': ['axis (pixels)',
  'dgt1 (pixels)',
  'dgt_holes (pixels)',
  'pixels_to_inches (inches)',
  'pixels_to_cm (centimeters)']}

**Why ~20,680 timestamps?**

The timestamp table has one row per **unique** event coordinate in the hierarchy:
- Parent timeline (dgt1) contains 30,092 hole events at absolute pixel coordinates
- But many holes share the same pixel row (multiple notes at one time position)
- After de-duplication: ~20,676 unique coordinates plus 4 boundary coordinates

Each row shows the coordinate in **all** coordinate systems simultaneously:
- `axis (pixels)`: The root (parent) coordinate in pixels
- `dgt1 (pixels)`: Same as axis (this IS the root timeline)
- `dgt_holes (pixels)`: **Relative** coordinate in child timeline (first hole = 0, NaN if outside)
- C-Map columns: Physical units with names like "pixels_to_inches (inches)"

In [8]:
timestamps_df.head(10)

,axis (pixels),dgt1 (pixels),dgt_holes (pixels),pixels_to_inches (inches),pixels_to_cm (centimeters)
id,,,,,
hole_K0_N1,15343,15343,0,51.100749,129.795903
hole_K0_N2,15391,15391,48,51.260616,130.201965
hole_K88_N1,15617,15617,274,52.013322,132.113838
hole_K86_N1,15626,15626,283,52.043297,132.189975
hole_K0_N1,15638,15638,295,52.083264,132.291490
hole_K88_N2,15655,15655,312,52.139883,132.435304
hole_K0_N2,15675,15675,332,52.206495,132.604496
hole_K88_N3,15683,15683,340,52.233139,132.672173
hole_K86_N2,15684,15684,341,52.236470,132.680633


Query a specific coordinate: at pixel 100,000 in the image, what's the local
coordinate in the holes region?

In [9]:
ts = dgt1.get_timestamp(100000.0)

{
    "Query (parent coord)": 100000.0,
    "Child coord (dgt_holes)": ts.get_coordinate_for("dgt_holes"),
    "Calculation": f"100000 - {aton_loader.first_hole.value} = {100000 - aton_loader.first_hole.value}",
}

{'Query (parent coord)': 100000.0,
 'Child coord (dgt_holes)': IdCoordinate(84657, pixels, 'dgt_holes'),
 'Calculation': '100000 - 15343 = 84657'}

Boundary table shows where child timelines start and end (with C-Maps).
Use `timestamp_table_to_dataframe()` for column names with units.

In [10]:
boundary_table = dgt1.get_boundary_table(conversion_maps=True)
boundary_df = timestamp_table_to_dataframe(boundary_table)
boundary_df

,axis (pixels),dgt1 (pixels),dgt_holes (pixels),pixels_to_inches (inches),pixels_to_cm (centimeters)
0,0,0,<NA>,0.000000,0.000000
1,15343,15343,0,51.100749,129.795903
2,293119,293119,277776,976.249792,2479.674471
3,299400,299400,<NA>,997.169026,2532.809326


***

## Part E: Load External Data Files

Now we load the MIDI files, audio, and score annotations. Each becomes a
separate timeline that we'll connect via the TimelineGroup.

**All loaders use the `from_file()` constructor for clean one-line loading.**

### E.1: DLT1 - Raw MIDI (one event per hole)

In [11]:
# One-line loading with from_file()
midi_raw_loader = PerformanceMidiLoader.from_file(MIDI_RAW_PATH)

# Create timeline directly from loader - no need to access store
dlt1_raw = midi_raw_loader.create_timeline(uid="dlt1_raw")
raw_note_count = len(midi_raw_loader.store.notes)

dlt1_raw

DiscreteLogicalTimeline(id='dlt1_raw', length=277776, unit=ticks, events=0, children=2)

In [12]:
{
    "DLT1 (MIDI Raw)": dlt1_raw.id,
    "Length": f"{dlt1_raw.length.value:,} ticks",
    "Note events": f"{raw_note_count:,}",
}

{'DLT1 (MIDI Raw)': 'dlt1_raw',
 'Length': '277,776 ticks',
 'Note events': '30,092'}

### E.2: DLT2 - Expressive MIDI (merged notes + dynamics)

In [13]:
midi_exp_loader = PerformanceMidiLoader.from_file(MIDI_EXP_PATH)
dlt2_exp = midi_exp_loader.create_timeline(uid="dlt2_exp")
exp_note_count = len(midi_exp_loader.store.notes)

dlt2_exp

DiscreteLogicalTimeline(id='dlt2_exp', length=274318, unit=ticks, events=0, children=2)

In [14]:
{
    "DLT2 (MIDI Expressive)": dlt2_exp.id,
    "Length": f"{dlt2_exp.length.value:,} ticks",
    "Note events": f"{exp_note_count:,}",
}

{'DLT2 (MIDI Expressive)': 'dlt2_exp',
 'Length': '274,318 ticks',
 'Note events': '6,380'}

### E.3: DPT1 - Audio (MP3)

**Note:** MP3 loading requires `mutagen` or `soundfile`. If not installed,
we use a mock timeline with the known duration from the README.

In [15]:
try:
    audio_loader = AudioLoader.from_file(MP3_PATH)
    dpt1_audio = audio_loader.create_timeline(uid="dpt1_audio")
    audio_duration = audio_loader.duration_seconds
    audio_info = {
        "DPT1 (Audio)": dpt1_audio.id,
        "Sample rate": f"{audio_loader.sample_rate:,} Hz",
        "Duration": f"{audio_duration:.2f} seconds",
    }
except ValueError as e:
    print(f"Note: MP3 loading unavailable ({e}). Using mock timeline.")
    audio_duration = 573.0
    dpt1_audio = ContinuousPhysicalTimeline(
        length=audio_duration,
        unit=TimeUnit.seconds,
        uid="dpt1_audio",
    )
    audio_info = {
        "DPT1 (Audio)": dpt1_audio.id,
        "Duration": f"{audio_duration:.2f} seconds (from README)",
    }

dpt1_audio

DiscretePhysicalTimeline(id='dpt1_audio', length=19670082, unit=samples, events=0, children=0, cmaps=1)

In [16]:
audio_info

{'DPT1 (Audio)': 'dpt1_audio',
 'Sample rate': '44,100 Hz',
 'Duration': '446.03 seconds'}

### E.4: CLT1 - Score Annotations (DCML TSV files)

The DCML corpus provides score data in TSV format. We load **all four files**
(notes, measures, harmonies, chords) at once using `from_file()` with glob.

**Important:** TimeToAlign! properly loads:
- `.harmonies.tsv` as **annotations** (Roman numeral analysis)
- `.chords.tsv` as **control events** (chord symbols)

In [17]:
SCORE_BASE = "WWV096-Meistersinger_01_Vorspiel-Prelude_SchottKleinmichel"
score_tsv_files = sorted(DCML_DIR.glob(f"{SCORE_BASE}.*.tsv"))

[f.name for f in score_tsv_files]

['WWV096-Meistersinger_01_Vorspiel-Prelude_SchottKleinmichel.chords.tsv',
 'WWV096-Meistersinger_01_Vorspiel-Prelude_SchottKleinmichel.harmonies.tsv',
 'WWV096-Meistersinger_01_Vorspiel-Prelude_SchottKleinmichel.measures.tsv',
 'WWV096-Meistersinger_01_Vorspiel-Prelude_SchottKleinmichel.notes.tsv']

Load all TSV files at once using `from_file()` with `*` unpacking - one line.

In [18]:
score_loader = Ms3Loader.from_file(*score_tsv_files)
score_loader.store.summary()

{'tables': {'notes': {'unit': 'quarters',
   'count': 5577,
   'range': (0.0, 884.0)},
  'measures': {'unit': 'quarters', 'count': 222, 'range': (0.0, 884.0)},
  'controls': {'unit': 'quarters', 'count': 4378, 'range': (0.0, 884.0)},
  'annotations': {'unit': 'quarters', 'count': 1074, 'range': (0.0, 879.5)}},
 'facts': {'has_rests': False,
  'format': 'tsv',
  'parser': 'ms3',
  'source': '/home/laser/git/tta/timetoalign/tests/data/supra/dcml/WWV096-Meistersinger_01_Vorspiel-Prelude_SchottKleinmichel.notes.tsv',
  'control_type': 'Chord',
  'n_controls': 4378,
  'annotation_type': 'Harmony',
  'n_annotations': 1074,
  'n_measures': 222,
  'flow_control': {'total_measures': 222,
   'has_repeats': False,
   'has_voltas': False,
   'has_breaks': False,
   'repeat_starts': 0,
   'repeat_ends': 0,
   'voltas': set(),
   'breaks': set()}}}

Verify against gold standard (ZERO TOLERANCE).

In [19]:
notes_count = len(score_loader.store.notes)
measures_count = len(score_loader.store.measures)
annotations_count = len(score_loader.store.annotations)  # Harmonies
controls_count = len(score_loader.store.controls)  # Chords

assert notes_count == 5577, f"Notes mismatch: {notes_count} != 5577"
assert measures_count == 222, f"Measures mismatch: {measures_count} != 222"

{
    "Notes": notes_count,
    "Measures": measures_count,
    "Annotations (harmonies)": annotations_count,
    "Controls (chords)": controls_count,
    "Verification": "PASSED",
}

{'Notes': 5577,
 'Measures': 222,
 'Annotations (harmonies)': 1074,
 'Controls (chords)': 4378,
 'Verification': 'PASSED'}

Create the score timeline using `create_timeline()` directly from the loader.

In [20]:
clt1_score = score_loader.create_timeline(uid="clt1_score")
clt1_score

ContinuousLogicalTimeline(id='clt1_score', length=888, unit=quarters, events=0, children=4, cmaps=2)

### E.5: Inspect Harmony Annotations and Chord Controls

- **Harmonies** are loaded as **annotations** (Roman numeral analysis)
- **Chords** are loaded as **control events** (chord symbols)

Let's examine the harmonies - we'll use a specific label for coordinate transfer!

In [21]:
# Get harmony annotations (from .harmonies.tsv)
harmonies = score_loader.store.annotations.filter(subtype="Harmony")
harmonies_df = harmonies.to_dataframe()[["name", "text", "start", "mc", "mn"]].head(20)
harmonies_df

,name,text,start,mc,mn
0,C.I,C.I,0,1,1
1,IM2,IM2,2,1,1
2,vi7,vi7,5,2,2
3,V7/ii,V7/ii,7,2,2
4,V2(6)/ii,V2(6)/ii,8,3,3
5,V2,V2,9,3,3
6,I6,I6,10,3,3
7,V43,V43,11,3,3
8,I,I,12,4,4
9,IV64,IV64,13,4,4


In [22]:
# Get chord control events (from .chords.tsv)
chords = score_loader.store.controls.filter(subtype="Chord")
if len(chords) > 0:
    chords_df = chords.to_dataframe()[["name", "text", "start", "mc", "mn"]].head(10)
    chords_df
else:
    {"Chord controls": "No chords loaded (file may not exist)"}

***

## Part F: Create the TimelineGroup

Now we bring everything together in a **TimelineGroup**. This establishes
commensurability between all timelines, enabling coordinate transfer.

**Important:** A TimelineGroup does NOT have a root timeline - all timelines
are peers. We create it from a list of timelines.

**Alignment structure:**
```
dgt_holes (pixels)
    |
    +-- dlt1_raw (MIDI ticks)
    |
    +-- dlt2_exp (MIDI ticks)
    |
    +-- dpt1_audio (seconds)
    |
    +-- clt1_score (quarterbeats)
```

In [23]:
group = TimelineGroup(
    id="supra_alignment",
    name="SUPRA Piano Roll Alignment",
    timelines=[dgt_holes, dlt1_raw, dlt2_exp, dpt1_audio, clt1_score],
)

group

TimelineGroup(id='supra_alignment', n_timelines=5, n_timestamps=2, locked=False)

In [24]:
{"Group": group.name, "Timelines": group.timeline_ids, "Count": group.n_timelines}

{'Group': 'SUPRA Piano Roll Alignment',
 'Timelines': ['dgt_holes',
  'dlt1_raw',
  'dlt2_exp',
  'dpt1_audio',
  'clt1_score'],
 'Count': 5}

***

## Part G: Group Timestamps - The Heart of Alignment

**This is the key feature!** The TimelineGroup maintains a timestamp table
where each row represents a synchronized point across ALL timelines.
Coordinate transfer uses these timestamps via interpolation.

In [25]:
# Get the full timestamp table
group_timestamps = group.to_dataframe()
group_timestamps

,dgt_holes (pixels),dlt1_raw (ticks),dlt2_exp (ticks),dpt1_audio (samples),clt1_score (quarters),samples_to_seconds (seconds),quarters_to_ticks (ticks),quarters_to_measures (floating_measures)
0,0,0,0,0,0,0.000000,0,1.0
1,277776,277776,274318,19670082,888,446.033605,426240,223.0


Each row shows the **same musical moment** in all coordinate systems.
Column names include units (e.g., "dgt_holes (pixels)", "dpt1_audio (seconds)").

***

## Part H: Coordinate Transfer Using Timestamps

**The right way to transfer coordinates is through timestamps!**
Instead of asking for one target coordinate at a time with
`get_coordinate_at()`, we use `get_timestamp_at()`, which returns a full
cross-section through all timelines in a single call.

### H.1: Transfer a Specific Harmony Label Across All Timelines

Let's take the first occurrence of a I chord (C major) and find its position
in every timeline.

In [26]:
# Find the first I chord (tonic) harmony
i_chords = score_loader.store.annotations.filter(text="I")
if len(i_chords) > 0:
    first_i = i_chords.to_dataframe().iloc[0]
    i_chord_qb = float(first_i["start"])
    i_chord_label = first_i["text"]
    i_chord_mc = first_i["mc"]
else:
    # Fallback if no I chord
    first_harmony = score_loader.store.annotations.to_dataframe().iloc[0]
    i_chord_qb = float(first_harmony["start"])
    i_chord_label = first_harmony["text"]
    i_chord_mc = first_harmony["mc"]

{
    "Harmony label": i_chord_label,
    "Position (quarterbeats)": i_chord_qb,
    "Measure": i_chord_mc,
}

{'Harmony label': 'I', 'Position (quarterbeats)': 12.0, 'Measure': np.int64(4)}

Now get the timestamp at this position - one call gives us ALL coordinates!

In [27]:
# Get timestamp at the harmony position in the score timeline
harmony_ts = group.get_timestamp_at(i_chord_qb, "clt1_score")
harmony_ts

ID,Coordinate,Type
clt1_score,12 quarters,axis
dgt_holes,3754 pixels,child
dlt1_raw,3754 ticks,child
dlt2_exp,3707 ticks,child
dpt1_audio,265812 samples,child
notes,3754 ticks,child
controls,3754 ticks,child
measures,12 quarters,child
annotations,12 quarters,child
ticks,5760 ticks,cmap


### H.2: Full Image Coordinates + Physical Units via IdCoordinate

The group timestamp gives us `dgt_holes` (relative pixels). To get the absolute
image position AND physical units (C-Maps), we use **IdCoordinate** - it carries
the child's timeline_id, so the parent automatically applies the offset!

In [28]:
if "dgt_holes" in harmony_ts.present_timelines:
    # The getter already returns an IdCoordinate carrying the child's timeline_id
    child_coord = harmony_ts.get_coordinate_for("dgt_holes")

    # Parent's to_dataframe() recognizes the child_id and applies offset
    image_ts = dgt1.to_dataframe(coordinates=[child_coord])
    image_ts

### H.3: Multiple Harmony Labels - Batch Transfer

For batch transfers, use `group.get_timestamps_at()` - the DEAD-SIMPLE API:
pass coordinates, get a DataFrame with all timelines and units in column names.

In [29]:
# Get first 10 harmonies
first_10_harmonies = (
    score_loader.store.annotations.filter(subtype="Harmony").to_dataframe().head(10)
)

# DEAD-SIMPLE: Get timestamps for all harmony coordinates in ONE CALL
harmony_coords = first_10_harmonies["start"].tolist()
group_df = group.get_timestamps_at(harmony_coords, "clt1_score")
group_df

,clt1_score (quarters),dgt_holes (pixels),dlt1_raw (ticks),dlt2_exp (ticks),dpt1_audio (samples),notes (ticks),controls (ticks),measures (quarters),annotations (quarters),ticks,floating_measures,seconds
0,0,0,0,0,0,0,0,0,0,0,1.00,0.000000
1,2,626,626,618,44302,626,626,2,2,960,1.50,1.004580
2,5,1564,1564,1545,110755,1564,1564,5,5,2400,2.25,2.511451
3,7,2190,2190,2162,155057,2190,2190,7,7,3360,2.75,3.516032
4,8,2502,2502,2471,177208,2502,2502,8,8,3840,3.00,4.018322
5,9,2815,2815,2780,199359,2815,2815,9,9,4320,3.25,4.520612
6,10,3128,3128,3089,221510,3128,3128,10,10,4800,3.50,5.022902
7,11,3441,3441,3398,243661,3441,3441,11,11,5280,3.75,5.525193
8,12,3754,3754,3707,265812,3754,3754,12,12,5760,4.00,6.027483
9,13,4067,4067,4016,287963,4067,4067,13,13,6240,4.25,6.529773


To also get the parent's C-Maps (inches, cm), pass IdCoordinates to the parent:

In [30]:
# Get dgt_holes coordinates from group timestamps as IdCoordinates
dgt_holes_col = (
    "dgt_holes (pixels)" if "dgt_holes (pixels)" in group_df.columns else "dgt_holes"
)
child_coords = [
    IdCoordinate(v, TimeUnit.pixels, "dgt_holes")
    for v in group_df[dgt_holes_col].dropna()
]

# Parent timeline auto-applies offset and returns timestamps with C-Maps
parent_df = dgt1.to_dataframe(coordinates=child_coords)
parent_df

,axis (pixels),dgt1 (pixels),dgt_holes (pixels),pixels_to_inches (inches),pixels_to_cm (centimeters)
0,15343,15343,0,51.100749,129.795903
1,15969,15969,626,53.185679,135.091624
2,16907,16907,1564,56.309742,143.026744
3,17533,17533,2190,58.394671,148.322465
4,17845,17845,2502,59.433805,150.961865
5,18158,18158,2815,60.476270,153.609725
6,18471,18471,3128,61.518734,156.257585
7,18784,18784,3441,62.561199,158.905445
8,19097,19097,3754,63.603664,161.553306
9,19410,19410,4067,64.646128,164.201166


***

## Part I: Comprehensive Group Timestamps Demo

Let's demonstrate all the capabilities of group timestamps.

### I.1: Query from Different Timelines

We can query the group from ANY member timeline. Just display the timestamp!

In [31]:
# Query at 100 seconds in the audio - timestamp shows ALL peer timelines
audio_100s = group.get_timestamp_at(100.0, "dpt1_audio")
audio_100s

ID,Coordinate,Type
dpt1_audio,100 samples,axis
dgt_holes,1 pixels,child
dlt1_raw,1 ticks,child
dlt2_exp,1 ticks,child
clt1_score,14800/3278347 quarters,child
notes,1 ticks,child
controls,1 ticks,child
measures,14800/3278347 quarters,child
annotations,14800/3278347 quarters,child
seconds,0.0022675736961451248 seconds,cmap


In [32]:
# Query at 50,000 pixels in the holes region
holes_50k = group.get_timestamp_at(50000, "dgt_holes")
holes_50k

ID,Coordinate,Type
dgt_holes,50000 pixels,axis
dlt1_raw,50000 ticks,child
dlt2_exp,49378 ticks,child
dpt1_audio,3540637 samples,child
clt1_score,925000/5787 quarters,child
notes,50000 ticks,child
controls,50000 ticks,child
measures,925000/5787 quarters,child
annotations,925000/5787 quarters,child
seconds,80.28655328798186 seconds,cmap


### I.2: Boundary Points

Check the alignment at the start and end of the musical content.
Display timestamps directly to see all coordinate values:

In [33]:
# Start of music (coordinate 0 in dgt_holes)
start_ts = group.get_timestamp_at(0, "dgt_holes")
start_ts

ID,Coordinate,Type
dgt_holes,0 pixels,axis
dlt1_raw,0 ticks,child
dlt2_exp,0 ticks,child
dpt1_audio,0 samples,child
clt1_score,0 quarters,child
notes,0 ticks,child
controls,0 ticks,child
measures,0 quarters,child
annotations,0 quarters,child
seconds,0 seconds,cmap


In [34]:
# End of music
end_ts = group.get_timestamp_at(int(aton_loader.musical_length.value), "dgt_holes")
end_ts

ID,Coordinate,Type
dgt_holes,277776 pixels,axis
dlt1_raw,277776 ticks,child
dlt2_exp,274318 ticks,child
dpt1_audio,19670082 samples,child
clt1_score,888 quarters,child
controls,277776 ticks,child
seconds,446.03360544217685 seconds,cmap
ticks,426240 ticks,cmap
floating_measures,223 floating_measures,cmap


### I.3: Round-Trip Verification

Verify that coordinate transfer is reversible. We show the full timestamps
at each step so you can see all coordinates involved.

In [35]:
test_coord = 100000  # pixels in holes region

# Step 1: Get timestamp at our test coordinate
ts1 = group.get_timestamp_at(test_coord, "dgt_holes")
ts1

ID,Coordinate,Type
dgt_holes,100000 pixels,axis
dlt1_raw,100000 ticks,child
dlt2_exp,98755 ticks,child
dpt1_audio,7081275 samples,child
clt1_score,1850000/5787 quarters,child
notes,100000 ticks,child
controls,100000 ticks,child
measures,1850000/5787 quarters,child
annotations,1850000/5787 quarters,child
seconds,160.5731292517007 seconds,cmap


In [36]:
# Step 2: Transfer to audio, then back to holes
ts2 = group.get_timestamp_at(ts1.get_coordinate_for("dpt1_audio"), "dpt1_audio")
ts2

ID,Coordinate,Type
dpt1_audio,7081275 samples,axis
dgt_holes,100000 pixels,child
dlt1_raw,100000 ticks,child
dlt2_exp,98755 ticks,child
clt1_score,1048028700/3278347 quarters,child
notes,100000 ticks,child
controls,100000 ticks,child
measures,1048028700/3278347 quarters,child
annotations,1048028700/3278347 quarters,child
seconds,160.5731292517007 seconds,cmap


In [37]:
# Step 3: Transfer to score, then back to holes
ts3 = group.get_timestamp_at(ts1.get_coordinate_for("clt1_score"), "clt1_score")
ts3

ID,Coordinate,Type
clt1_score,1850000/5787 quarters,axis
dgt_holes,100000 pixels,child
dlt1_raw,100000 ticks,child
dlt2_exp,98755 ticks,child
dpt1_audio,7081275 samples,child
notes,100000 ticks,child
controls,100000 ticks,child
measures,1850000/5787 quarters,child
annotations,1850000/5787 quarters,child
ticks,153447 ticks,cmap


In [38]:
# Verify round-trip precision by examining coordinates from each timestamp
via_audio = ts2.get_coordinate_for("dgt_holes", format="int")
via_score = ts3.get_coordinate_for("dgt_holes", format="int")
print("Round-trip verification:")
print(f"  Original:   {test_coord} pixels")
print(f"  Via audio:  {via_audio} pixels")
print(f"  Via score:  {via_score} pixels")
print(f"  Audio diff: {abs(via_audio - test_coord)} pixels")
print(f"  Score diff: {abs(via_score - test_coord)} pixels")

Round-trip verification:
  Original:   100000 pixels
  Via audio:  100000 pixels
  Via score:  100000 pixels
  Audio diff: 0 pixels
  Score diff: 0 pixels


***

## Summary

In this tutorial, we demonstrated a complete alignment workflow:

1. **Image Timeline (DGT1)**: Created from ATON analysis with holes as events
2. **Physical C-Maps**: Attached with human-readable names (`name=`)
3. **Child Timeline (dgt_holes)**: Modeled the musical region as a child
4. **Timestamps**: Showed cross-section views through the hierarchy
5. **External Data**: Loaded MIDI, audio, and score with harmonies + chords
6. **TimelineGroup**: Connected all timelines (no root - all peers)
7. **Coordinate Transfer via Timestamps**: The RIGHT way to transfer coordinates!

### Key Patterns

| Pattern | Usage |
|---------|-------|
| One-line loading | `ATONLoader.from_file(path)` |
| Direct timeline creation | `loader.create_timeline(uid=...)` |
| Named C-Maps | `ScalarMap(..., name="pixels_to_inches")` |
| Coordinate transfer | `group.get_timestamp_at(coord, timeline_id)` |
| Timeline display | Just `timeline` (no `print()` needed) |

### Timeline Diagram

```
DGT1 (Full Image: 0 - 299,400 px)
  |-- pixels_to_inches (C-Map)
  |-- pixels_to_cm (C-Map)
  |
  +-- [15,343 px] -- dgt_holes (Musical Region: 0 - 277,776 px) -- [293,119 px]
                          |
                          | TimelineGroup (all peers)
                          |
                          +-- dlt1_raw (MIDI raw: ticks)
                          |
                          +-- dlt2_exp (MIDI expressive: ticks)
                          |
                          +-- dpt1_audio (Audio: seconds)
                          |
                          +-- clt1_score (Score: quarterbeats)
                                |-- notes (5,577 events)
                                |-- measures (222 events)
                                |-- annotations (harmonies from .harmonies.tsv)
                                +-- controls (chords from .chords.tsv)
```

## Next Steps

- **how01_beat_grids.ipynb**: Work with BeatGrid, FloorMap, and RotationMap
- **Advanced**: Implement WarpMap for non-linear alignment (expressive timing)